In [1]:
%%configure -f
{ "conf": {
          "spark.jars": " s3://ppc-edm-prod-emr/common_jar/greenplum-connector-apache-spark-scala_2.12-2.1.3.jar,s3://ppc-edm-prod-emr/common_jar/ojdbc6.jar,s3://ppc-edm-prod-emr/common_jar/elasticsearch-spark-30_2.12-7.17.3.jar",
          "spark.app.name" : "Data Tape Validation"
    }
}

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2601,application_1781643798877_0348,pyspark,idle,Link,Link,None,


In [2]:
# %%configure -f
# { "conf": {
#           "spark.jars": " s3://ppc-edm-prod-emr/common_jar/greenplum-connector-apache-spark-scala_2.12-2.1.3.jar,s3://ppc-edm-prod-emr/common_jar/ojdbc6.jar,s3://ppc-edm-prod-emr/common_jar/elasticsearch-spark-30_2.12-7.17.3.jar",
#           "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension",
#           "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog",
#           "spark.jars.packages": "io.delta:delta-core_2.12:0.7.0,com.crealytics:spark-excel_2.12:0.13.7",
#           "spark.app.name" : "Data Tape Validation"
#     }
# }

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2602,application_1781643798877_0349,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
url = "jdbc:postgresql://edm_gp_prod.edm.ppc:6432/edmprod"
user = "edm_load"
passwd = "3dm_10a6"

def data_extract(query):
    return spark.read.format('jdbc').option('url',url).option("driver",'org.postgresql.Driver').option('user',user).option('password',passwd).option(
                'query', query).load()

def data_extract_oracle(query):
    return spark.read.format('jdbc').option('url',"jdbc:oracle:thin:EDM_PIPE/k33p$Out@oda-prodfin.purchasingpwr.com:1521/PRODFIN").option("driver","oracle.jdbc.OracleDriver").option("fetchsize", "100000").option('user',"EDM_PIPE").option('password',"k33p$Out").option(
                'query', query).load()

def data_extract_RPT(query):
    return spark.read.format('jdbc').option('url',"jdbc:oracle:thin:ETLUSER/etlrpt12@oda-prod.purchasingpwr.com:1521/RPTMT").option("driver","oracle.jdbc.OracleDriver").option("fetchsize", "100000").option('user',"ETLUSER").option('password',"etlrpt12").option(
                'query', query).load()


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
gp_sold_contract = data_extract("""select  orderid,SOLDTO, SOLDAMT, SOLDDATE, SOLDPOOL  from ods.xxcon_contract_stage
"""
)
gp_sold_contract.persist()
gp_sold_contract.count()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
Ora_order_events = data_extract_oracle("""select orderid,SOLDTO, SOLDAMT, SOLDDATE, SOLDPOOL from xxcontract.xxcon_contract""")
Ora_order_events.persist()
Ora_order_events.count()

In [ ]:
gp_sold_contract.createOrReplaceTempView("gpcontract")

In [ ]:
Ora_order_events.createOrReplaceTempView("oratape")


In [ ]:
gp_sold_tape = data_extract(
"""select orderid,soldto,soldpool,soldamt from ods.treasury_datatape_daily tdd  where as_of_date =current_date - 1
"""
)
gp_sold_tape.persist()
gp_sold_tape.count()

In [ ]:
gp_sold_tape.createOrReplaceTempView("gptape")



In [13]:
gp_sold_tape.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------+------+-----------------+-------+
| orderid|soldto|         soldpool|soldamt|
+--------+------+-----------------+-------+
|17565022| GRP18|2018.A.02.20.2018| 534.92|
|23872799| GRP18|2018.A.12.10.2018|2391.62|
|26327125| GRP18|2018.A.2019.05.08| 466.48|
|28994233| GRP18|2018.A.2019.12.31| 504.18|
|29399717|   PPC|             null|   null|
+--------+------+-----------------+-------+
only showing top 5 rows

In [ ]:
Ora_order_events.show()

In [15]:
# sql="""select
# g.orderid,o.orderid
# ,g.soldto,o.soldto
# ,g.soldamt,o.soldamt
# ,g.soldpool,o.soldpool
# from 
# gpcontract g join oratape o
# on g.orderid=o.orderid
# where 
# (g.soldto!=o.soldto) or
# (g.soldamt!=o.soldamt) or
# (g.soldpool!=o.soldpool)
# and o.orderid in (select orderid from gptape)
# """


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
# sql="""select
# g.orderid,o.orderid
# ,g.soldto,o.soldto
# ,g.soldamt,o.soldamt
# ,g.soldpool,o.soldpool
# from 
# gpcontract g join gptape o
# on g.orderid=o.orderid
# where 
# (g.soldto!=o.soldto) or
# (g.soldamt!=o.soldamt) or
# (g.soldpool!=o.soldpool)
# """


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [17]:
# sql="""select
# g.orderid,o.orderid
# ,g.soldto,o.soldto
# ,g.soldamt,o.soldamt
# ,g.soldpool,o.soldpool
# from 
# gpcontract g join oratape o
# on g.orderid=o.orderid
# where 
# (g.soldto!=o.soldto) or
# (g.soldamt!=o.soldamt) or
# (g.soldpool!=o.soldpool)"""

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
sql="""select
g.orderid,o.orderid
,g.soldto,o.soldto
,g.soldamt,o.soldamt
,g.soldpool,o.soldpool
from 
gptape g join oratape o
on g.orderid=o.orderid
where 
(g.soldto!=o.soldto) or
(g.soldamt!=o.soldamt) or
(g.soldpool!=o.soldpool)
"""

In [ ]:
res=spark.sql(sql)
res.cache()

In [ ]:
res.count()
#res.show()

In [ ]:
res.select("g.orderid").distinct().show()

In [23]:
# output_path = "s3://ppc-edm-dev/harimu/rough/treasury_issue_2026_06_05/"

# res.select("g.orderid").distinct().coalesce(1).write \
#     .mode("overwrite") \
#     .option("header", "true") \
#     .csv(output_path)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…